# Agentic Vacuum-World Simulation

This notebook simulates a **vacuum-cleaning agent** moving through a 2x2 grid of rooms, cleaning any room that is dirty.

Run each cell below **from top to bottom, in order** (Shift+Enter).

### Step 1: Import required libraries

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams["figure.max_open_warning"] = 0  # suppress a harmless warning
%matplotlib inline

### Step 2: Settings (Configuration)

Control the simulation from here:
- `AGENT_TYPE` — which agent to run
- `STEPS` — how many steps the simulation runs
- `DIRT_RESPAWN_PROB` — chance a clean room becomes dirty again each step
- `SAVE_GIF` — whether to save the animation as a GIF

In [ ]:
AGENT_TYPE = "utility_based"     # options: "simple_reflex" | "model_based_reflex" | "utility_based"
STEPS = 20
DIRT_RESPAWN_PROB = 0.15
SAVE_GIF = True
RANDOM_SEED = 7

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

### Step 3: Define the environment (rooms)

In [ ]:
environment = {
    "Room1": "Clean",
    "Room2": "Dirty",
    "Room3": "Clean",
    "Room4": "Clean",
}

room_positions = {
    "Room1": (0, 1),  # Top-left
    "Room2": (1, 1),  # Top-right
    "Room3": (0, 0),  # Bottom-left
    "Room4": (1, 0),  # Bottom-right
}

rooms = list(environment.keys())
agent_index = 0

print(environment)

### Step 4: Agent memory (internal model) and performance stats

In [ ]:
agent_model = {room: None for room in rooms}   # None = not perceived yet

history = []
stats = {"cleans": 0, "moves": 0, "wasted_moves": 0}

def distance(room_a, room_b):
    ax, ay = room_positions[room_a]
    bx, by = room_positions[room_b]
    return abs(ax - bx) + abs(ay - by)

### Step 5: Agent 1 — Simple Reflex Agent

Only looks at the current room. If dirty, clean it; otherwise move to the next room.

In [ ]:
def simple_reflex_agent(state, current_room):
    if state == "Dirty":
        return "Clean", None
    return "Move", (agent_index + 1) % len(rooms)

### Step 6: Agent 2 — Model-Based Reflex Agent

Keeps a memory of every room it has seen, so it can skip rooms already known to be clean.

In [ ]:
def model_based_reflex_agent(state, current_room):
    agent_model[current_room] = state

    if state == "Dirty":
        return "Clean", None

    for step in range(1, len(rooms) + 1):
        candidate_idx = (agent_index + step) % len(rooms)
        candidate = rooms[candidate_idx]
        if agent_model[candidate] != "Clean":
            return "Move", candidate_idx

    return "Move", (agent_index + 1) % len(rooms)

### Step 7: Agent 3 — Utility-Based Agent

The most "intelligent" of the three — looks at its memory, finds the nearest known dirty room, and heads straight there (goal-directed).

In [ ]:
def utility_based_agent(state, current_room):
    agent_model[current_room] = state

    if state == "Dirty":
        return "Clean", None

    known_dirty = [r for r, s in agent_model.items() if s == "Dirty"]

    if known_dirty:
        target = min(known_dirty, key=lambda r: distance(current_room, r))
        target_idx = rooms.index(target)
        return "Move", target_idx

    unknown = [r for r, s in agent_model.items() if s is None]
    if unknown:
        target_idx = rooms.index(unknown[0])
        return "Move", target_idx

    return "Move", (agent_index + 1) % len(rooms)

### Step 8: Put all three agents into a dictionary (so AGENT_TYPE can select one)

In [ ]:
AGENTS = {
    "simple_reflex": simple_reflex_agent,
    "model_based_reflex": model_based_reflex_agent,
    "utility_based": utility_based_agent,
}

### Step 9: Visualization function

Draws the grid plus a side panel showing the agent's memory and stats.

In [ ]:
def draw_environment(env, agent_pos, step, action_taken):
    fig, (ax, info_ax) = plt.subplots(
        1, 2, figsize=(9, 4.5), gridspec_kw={"width_ratios": [1.2, 1]}
    )

    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Step {step} — Agent in {rooms[agent_pos]} — Action: {action_taken}")

    for room, pos in room_positions.items():
        x, y = pos
        color = "red" if env[room] == "Dirty" else "green"
        rect = patches.Rectangle((x, y), 1, 1, facecolor=color, edgecolor="black")
        ax.add_patch(rect)
        ax.text(x + 0.5, y + 0.8, room, ha="center", va="center", color="white", fontsize=10)
        ax.text(x + 0.5, y + 0.3, env[room], ha="center", va="center", color="white", fontsize=8)

    agent_x, agent_y = room_positions[rooms[agent_pos]]
    ax.add_patch(patches.Circle((agent_x + 0.5, agent_y + 0.5), 0.12, color="blue", zorder=5))

    legend_elems = [
        patches.Patch(facecolor="red", label="Dirty"),
        patches.Patch(facecolor="green", label="Clean"),
        patches.Circle((0, 0), 0.1, color="blue", label="Agent"),
    ]
    ax.legend(handles=legend_elems, loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=8)

    info_ax.axis("off")
    info_ax.set_title(f"Agent: {AGENT_TYPE}", fontsize=11, loc="left")

    model_lines = []
    for room in rooms:
        belief = agent_model.get(room)
        belief_str = belief if belief is not None else "unknown"
        model_lines.append(f"  {room}: {belief_str}")

    total_actions = stats["cleans"] + stats["moves"]
    efficiency = (stats["cleans"] / total_actions * 100) if total_actions else 0.0

    text = (
        "Internal model (memory):\n" + "\n".join(model_lines) +
        "\n\nPerformance:\n"
        f"  Cleans: {stats['cleans']}\n"
        f"  Moves:  {stats['moves']}\n"
        f"  Efficiency: {efficiency:.0f}% (cleans / total actions)\n\n"
        "Recent history:\n" +
        "\n".join(
            f"  {h['step']}: {h['room']} [{h['percept']}] -> {h['action']}"
            for h in history[-6:]
        )
    )
    info_ax.text(0.0, 1.0, text, va="top", ha="left", fontsize=8.5, family="monospace")

    plt.tight_layout()
    return fig

### Step 10: Helper function for GIF export

In [ ]:
def _fig_to_array(fig):
    import numpy as np
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    arr = np.asarray(buf)
    return arr

### Step 11: Simulation loop

The main logic — each step the agent perceives, decides, acts, and the environment updates (dirt can also randomly reappear).

In [ ]:
def run_simulation():
    global agent_index

    agent_fn = AGENTS[AGENT_TYPE]
    frames = []

    for step in range(1, STEPS + 1):
        current_room = rooms[agent_index]
        state = environment[current_room]

        action, next_index = agent_fn(state, current_room)

        if action == "Clean":
            environment[current_room] = "Clean"
            agent_model[current_room] = "Clean"
            stats["cleans"] += 1
        else:
            if next_index == agent_index:
                stats["wasted_moves"] += 1
            agent_index = next_index
            stats["moves"] += 1

        history.append({"step": step, "room": current_room, "percept": state, "action": action})

        fig = draw_environment(environment, agent_index if action == "Move" else rooms.index(current_room), step, action)
        frames.append(fig)
        plt.show()
        if not SAVE_GIF:
            plt.close(fig)

        for room in rooms:
            if environment[room] == "Clean" and random.random() < DIRT_RESPAWN_PROB:
                environment[room] = "Dirty"

    if SAVE_GIF and frames:
        import matplotlib.animation as animation

        gif_fig, gif_ax = plt.subplots(figsize=(9, 4.5))

        def _make_frame(i):
            gif_ax.clear()
            gif_ax.axis("off")
            gif_ax.imshow(_fig_to_array(frames[i]))

        ani = animation.FuncAnimation(gif_fig, _make_frame, frames=len(frames), interval=400)
        out_path = "simulation.gif"
        ani.save(out_path, writer="pillow", fps=2)
        plt.close(gif_fig)
        for f in frames:
            plt.close(f)
        print(f"Saved animation to {out_path}")

    print("\nSimulation complete!")
    print(f"Agent type: {AGENT_TYPE}")
    print(f"Total cleans: {stats['cleans']}, total moves: {stats['moves']}, wasted moves: {stats['wasted_moves']}")

### Step 12: Run the simulation

Running this cell will display the animation frame-by-frame below, and save a `simulation.gif` file.

In [ ]:
run_simulation()

### (Optional) Step 13: Try a different agent

To see a different agent type (e.g. `simple_reflex` or `model_based_reflex`):
1. Go back to the **Step 2 (Settings)** cell and change `AGENT_TYPE`
2. Re-run the **Step 4 (memory/stats)** cell to reset the agent's memory
3. Re-run the **Step 12 (run_simulation())** cell